# Cuaderno de estudio · El reporte sección por sección

Este cuaderno sigue **el mismo orden que el reporte PDF**. Cada sección tiene cuatro partes:

| Parte | Qué contiene |
|---|---|
| 📄 **Qué dice el reporte** | El resumen de la sección y sus tablas o gráficas |
| 💻 **De dónde salen los números** | El código que calcula cada número, con un comentario por instrucción |
| 🎤 **Qué decir en la exposición** | Las frases listas para presentar |
| ❓ **Preguntas del profesor** | Las preguntas probables, con su respuesta |

**Cómo estudiarlo en poco tiempo:**
1. Ejecuta todo (*Run All*).
2. Lee cada sección con el PDF abierto en la misma sección.
3. Memoriza las frases de 🎤 y las respuestas de ❓. Los números de las celdas de código son los mismos del PDF.

| Sección del reporte | Tema | Tablas y gráficas |
|---|---|---|
| 1 | Planteamiento del problema | Tabla 2 |
| 2 | Población y muestra | Tablas 3 y 4 |
| 3 | Recolección y descripción | Tablas 5–7, Gráficas 1–5 |
| 4 | Estimación puntual | Tabla 8 |
| 5 | Estimación por intervalo | Tabla 9, Gráfica 6 |
| 6 | Análisis e interpretación | Gráficas 7–9 |
| 7 | Teoría de decisión | Tablas 10–11, Gráfica 10 |
| 8 | Decisiones metodológicas | — |
| — | Conclusiones y guion de la exposición | — |

In [1]:
# QUÉ HACE ESTA CELDA: carga las herramientas y los datos
# numpy: cálculos con números
import numpy as np
# pandas: tablas, como en Excel
import pandas as pd
# scipy.stats: valores z y t, pruebas estadísticas
from scipy import stats

# read_csv lee un archivo CSV y lo guarda como tabla
poblacion = pd.read_csv("../data/processed/poblacion_clientes.csv")
muestra = pd.read_csv("../data/processed/muestra_clientes_n1000.csv")
# len() cuenta las filas (clientes)
N = len(poblacion)
n = len(muestra)
print("Clientes en la población (N):", N)
print("Clientes en la muestra (n):", n)

Clientes en la población (N): 10000
Clientes en la muestra (n): 1000


---
# Sección 1 · Planteamiento del problema

## 📄 Qué dice el reporte
- **Problema:** la deserción (*churn*): clientes que cierran su cuenta y se van a otro banco.
- **Por qué importa:** cada cliente que se va se lleva sus depósitos y sus comisiones. Conseguir un cliente nuevo cuesta más que conservar uno.
- **Población de interés:** los 10,000 clientes del banco en Francia, Alemania y España.
- **Qué se quiere conocer (Tabla 2):**

| Pregunta | Qué se quiere conocer | Parámetro | Tipo |
|---|---|---|---|
| P1 | Qué proporción de clientes abandona | $p$ | Proporción |
| P2 | Saldo promedio de quienes abandonan | $\mu$ | Media |
| P3 | Cuánto más abandonan los inactivos que los activos | $p_1 - p_2$ | Diferencia de proporciones |
| P4 | Diferencia de edad promedio entre quien se va y quien se queda | $\mu_1 - \mu_2$ | Diferencia de medias |

**Parámetro:** un número que describe a **toda la población** (por ejemplo, el porcentaje real de abandono). No se conoce: se estima con la muestra.

## 🎤 Qué decir en la exposición
> Estudiamos la deserción de clientes de un banco europeo con 10,000 clientes. Queremos conocer cuatro números de toda la cartera: qué proporción abandona, cuánto dinero tienen quienes abandonan, cuánto más abandonan los inactivos y cuántos años más tienen quienes abandonan. Con esos números decidimos a quién dirigir una campaña de retención.

## ❓ Preguntas del profesor
**¿Por qué cuatro preguntas?** El documento pide al menos dos técnicas de estimación entre media, diferencia de medias, proporción y diferencia de proporciones. Usamos las cuatro, una por pregunta.

---
# Sección 2 · Identificación de población y muestra

## 📄 Qué dice el reporte

| Subsección | Lo que hay que saber |
|---|---|
| 2.1 Población y calidad | Se revisaron 12 reglas de calidad (sin faltantes, sin duplicados, rangos válidos). Se cumplieron las 12. El 36 % tiene saldo 0 y se conserva porque es real. N = 10,000 |
| 2.2 ¿Por qué una muestra? | En un banco real, confirmar cada abandono exige revisar un expediente: cuesta tiempo y dinero. Como tenemos a los 10,000, al final podemos comprobar si acertamos |
| 2.3 Tamaño de muestra (Tabla 3) | La fórmula pide ≈ 965 clientes; se tomaron 1,000 |
| 2.4 Método (Tabla 4) | Muestreo aleatorio simple sin reemplazo, semilla 2026. La prueba χ² confirma que la muestra se parece a la población |

### 2.3 Tamaño de la muestra

$$n_0 = \frac{z^2 \cdot p(1-p)}{E^2} \qquad n = \frac{n_0}{1 + \dfrac{n_0 - 1}{N}}$$

| Símbolo | Qué es | Valor | Por qué |
|---|---|---|---|
| $z$ | Valor crítico para 95 % de confianza | 1.96 | Nivel de confianza elegido |
| $p$ | Proporción esperada | 0.5 | No se conocía; 0.5 es el caso que pide más clientes |
| $E$ | Margen de error deseado | 0.03 | ±3 puntos basta para planear una campaña |
| $N$ | Tamaño de la población | 10,000 | La segunda fórmula corrige porque la población es finita |

In [2]:
# QUÉ HACE ESTA CELDA: calcula el tamaño de muestra (Tabla 3)
# norm.ppf(0.975) da el valor z que deja 2.5 % en cada orilla de la campana: 1.96
z = stats.norm.ppf(0.975)
p_supuesta = 0.5
margen_deseado = 0.03
# fórmula para población infinita (** 2 = al cuadrado)
n_inicial = z ** 2 * p_supuesta * (1 - p_supuesta) / margen_deseado ** 2
# corrección porque la población tiene sólo 10,000 clientes
n_corregido = n_inicial / (1 + (n_inicial - 1) / N)
print("z:", round(z, 4))
print("n0 (población infinita):", round(n_inicial, 1))
print("n corregido (N = 10,000):", round(n_corregido, 1))
print("n elegido:", n)
print("Fracción de muestreo n/N:", n / N)

z: 1.96
n0 (población infinita): 1067.1
n corregido (N = 10,000): 964.3
n elegido: 1000
Fracción de muestreo n/N: 0.1


**De dónde sale:** $n_0$ = 1.96² × 0.25 / 0.0009 = 1,067.1. Con la corrección, $n$ = 964.3, que se redondea a **965**. Se eligió **1,000**: número redondo y un colchón para los subgrupos de P3 y P4.

### 2.4 Representatividad: prueba χ² (Tabla 4)

La prueba responde: **¿las diferencias entre los % de la muestra y los de la población son del tamaño que produce el azar del sorteo?**
- **H0:** la distribución de la variable en la muestra es igual a la de la población.
- **H1:** es distinta.
- **α = 0.05.** Si p-valor > 0.05, no se rechaza H0: la muestra se parece a la población.

Se calcula como ejemplo la fila de país:

In [3]:
# QUÉ HACE ESTA CELDA: prueba χ² para PAÍS (fila 1 de la Tabla 4)
# value_counts cuenta cuántos hay de cada categoría; normalize=True lo da como proporción
proporcion_pais_poblacion = poblacion["pais"].value_counts(normalize=True)
# sort_index ordena las categorías para que población y muestra tengan el mismo orden
proporcion_pais_poblacion = proporcion_pais_poblacion.sort_index()
conteo_pais_muestra = muestra["pais"].value_counts()
conteo_pais_muestra = conteo_pais_muestra.sort_index()
# cuántos clientes de cada país esperaríamos con los mismos % que la población
esperado_pais = proporcion_pais_poblacion * n
# chisquare compara lo observado contra lo esperado
prueba_pais = stats.chisquare(conteo_pais_muestra, esperado_pais)
print("% población:", (proporcion_pais_poblacion * 100).round(1).to_dict())
print("% muestra:", (conteo_pais_muestra / n * 100).round(1).to_dict())
print("Estadístico χ²:", round(prueba_pais.statistic, 3))
print("p-valor:", round(prueba_pais.pvalue, 3))

% población: {'Alemania': 25.1, 'España': 24.8, 'Francia': 50.1}
% muestra: {'Alemania': 27.6, 'España': 22.7, 'Francia': 49.7}
Estadístico χ²: 4.279
p-valor: 0.118


**Tabla 4 completa** (cuaderno 02, sección 2.4):

| Variable | p-valor | Decisión |
|---|---|---|
| País | 0.12 | No se rechaza H0 |
| Género | 0.83 | No se rechaza H0 |
| Miembro activo | 0.17 | No se rechaza H0 |
| Número de productos | 0.43 | No se rechaza H0 |
| Grupo de edad | 0.92 | No se rechaza H0 |

**Conclusión:** No se rechaza H0 en ninguna variable (todos los p-valores > 0.05). Las diferencias entre muestra y población son del tamaño que produce el azar: la muestra representa a la población.

## 🎤 Qué decir en la exposición
> La población son los 10,000 clientes. Con 95 % de confianza y un margen de ±3 puntos, la fórmula pide 965 clientes; tomamos 1,000. Los sorteamos con muestreo aleatorio simple y semilla 2026, para que cualquiera pueda repetir la misma muestra. La prueba χ² dio p-valores entre 0.12 y 0.92 en país, género, actividad, productos y edad: la muestra se parece a la población.

## ❓ Preguntas del profesor
**¿Por qué no usaron a los 10,000 si los tenían?** En un banco real, confirmar cada abandono cuesta revisar un expediente. Además, tener a los 10,000 nos permite comprobar al final si los intervalos acertaron.

**¿Por qué p = 0.5 en la fórmula?** Porque no se conocía la tasa de abandono, y 0.5 es el valor que da la muestra más grande.

**¿Qué es la semilla?** El número con el que empieza el sorteo de la computadora. Con la misma semilla sale la misma muestra.

**¿Criterios de inclusión y exclusión?** Se incluyen clientes mayores de edad, con estatus de abandono conocido y de los tres países. Se excluyen duplicados, datos imposibles y datos personales innecesarios (el apellido). Ningún cliente se excluyó.

**¿No comparar la tasa de abandono en la prueba χ²?** No, porque es lo que queremos estimar.

---
# Sección 3 · Recolección y descripción de datos

## 📄 Qué dice el reporte
- **Cómo se obtuvieron:** dataset público *Churn Modelling* de Kaggle, revisado y preparado con Python (cuaderno 01).
- **Tabla 5 · cuantitativas:** media, mediana, desviación estándar, mínimo, máximo y rango.
- **Tabla 6 · cualitativas:** frecuencias y porcentajes.
- **Tabla 7 · contingencia:** actividad contra abandono.
- **Gráficas 1 a 5:** cada una con su párrafo «¿Qué aporta?».

In [4]:
# QUÉ HACE ESTA CELDA: medidas de la Tabla 5 para edad y saldo
# describe() calcula: count = cuántos, mean = media, std = desviación estándar, min, 50% = mediana, max
resumen_cuantitativas = muestra[["edad", "saldo"]].describe()
resumen_cuantitativas

,edad,saldo
count,1000.000000,1000.000000
mean,39.036000,75457.168330
std,10.505287,63151.030765
min,18.000000,0.000000
25%,32.000000,0.000000
50%,38.000000,95809.285000
75%,44.000000,128735.460000
max,79.000000,204223.030000


In [5]:
# QUÉ HACE ESTA CELDA: Tabla 7 · actividad contra abandono
# map cambia 0/1 por palabras
actividad = muestra["miembro_activo"].map({0: "Inactivo", 1: "Activo"})
estatus = muestra["abandono"].map({0: "Permanece", 1: "Abandona"})
# crosstab cuenta clientes en cada combinación; margins=True agrega totales
tabla_contingencia = pd.crosstab(actividad, estatus, margins=True, margins_name="Total")
tabla_contingencia

abandono,Abandona,Permanece,Total
miembro_activo,,,
Activo,83,454,537
Inactivo,123,340,463
Total,206,794,1000


| Lo que hay que recordar | Número | Gráfica |
|---|---|---|
| Clientes que abandonaron en la muestra | 206 de 1,000 (20.6 %) | Gráfica 1 |
| Abandono de inactivos contra activos | 26.6 % contra 15.5 % (123 de 463 y 83 de 537) | Gráfica 2 → origina P3 |
| Saldo | Media €75,457, mediana €95,809: la mediana es mayor porque muchos tienen saldo 0 | Gráfica 3: dos grupos (0 y ~100–150 mil €) |
| Edad | Media 39.0, mediana 38 | Gráfica 4: quienes abandonan son mayores → origina P4 |
| País | Alemania ≈ 28 % contra ≈ 18 % en Francia y España | Gráfica 5: hallazgo adicional |

## 🎤 Qué decir en la exposición
> En la muestra, 1 de cada 5 clientes abandonó. Los inactivos abandonan 26.6 % contra 15.5 % de los activos, y quienes abandonan son mayores. Estas diferencias se ven en la muestra; las secciones 5 y 6 dicen si existen en toda la población. El saldo forma dos grupos: muchos clientes con saldo cero y otro grupo entre 100 y 150 mil euros.

## ❓ Preguntas del profesor
**¿Por qué en el saldo la mediana es mayor que la media?** Porque el 36 % de los clientes tiene saldo 0, y esos ceros bajan el promedio. La mediana no se mueve tanto con valores extremos.

**¿Qué aporta la Gráfica 2?** Muestra que los inactivos abandonan más en la muestra. Eso genera la pregunta P3: ¿la diferencia existe en la población o fue el azar del sorteo?

---
# Sección 4 · Estimación puntual

## 📄 Qué dice el reporte
- **Parámetro:** el número de toda la población. No se conoce.
- **Estadístico:** el mismo número calculado con la muestra. Se escribe con gorro o barra ($\hat p$ "p gorro", $\bar x$ "x barra").
- **Estimación puntual:** usar ese único número como estimación del parámetro.
- El documento exige **interpretar**, no sólo dar el número.

In [6]:
# QUÉ HACE ESTA CELDA: las 4 estimaciones puntuales (Tabla 8)
# True para los clientes de cada grupo
es_abandono = muestra["abandono"] == 1
es_permanece = muestra["abandono"] == 0
es_inactivo = muestra["miembro_activo"] == 0
es_activo = muestra["miembro_activo"] == 1

# P1: el promedio de una columna de 0 y 1 es la proporción de 1
p_hat = muestra["abandono"].mean()
print("P1 · p̂:", p_hat)

# P2: loc[filas, columna] toma el saldo sólo de quienes abandonaron
saldo_abandona = muestra.loc[es_abandono, "saldo"]
media_saldo = saldo_abandona.mean()
print("P2 · x̄ del saldo (€):", round(media_saldo, 2))

# P3: proporción de abandono en cada grupo y su resta
abandono_inactivos = muestra.loc[es_inactivo, "abandono"]
abandono_activos = muestra.loc[es_activo, "abandono"]
p_inactivos = abandono_inactivos.mean()
p_activos = abandono_activos.mean()
dif_p = p_inactivos - p_activos
print("P3 · p̂ inactivos:", round(p_inactivos, 4))
print("P3 · p̂ activos:", round(p_activos, 4))
print("P3 · diferencia:", round(dif_p, 4))

# P4: edad promedio de cada grupo y su resta
edad_abandona = muestra.loc[es_abandono, "edad"]
edad_permanece = muestra.loc[es_permanece, "edad"]
media_edad_abandona = edad_abandona.mean()
media_edad_permanece = edad_permanece.mean()
dif_edad = media_edad_abandona - media_edad_permanece
print("P4 · edad promedio, abandona:", round(media_edad_abandona, 2))
print("P4 · edad promedio, permanece:", round(media_edad_permanece, 2))
print("P4 · diferencia (años):", round(dif_edad, 2))

P1 · p̂: 0.206
P2 · x̄ del saldo (€): 91561.53
P3 · p̂ inactivos: 0.2657
P3 · p̂ activos: 0.1546
P3 · diferencia: 0.1111
P4 · edad promedio, abandona: 46.5
P4 · edad promedio, permanece: 37.1
P4 · diferencia (años): 9.41


| Pregunta | Parámetro | Estadístico | Estimación | n |
|---|---|---|---|---|
| P1 | $p$ | $\hat p$ | 0.206 | 1,000 |
| P2 | $\mu$ | $\bar x$ | €91,562 | 206 |
| P3 | $p_1 - p_2$ | $\hat p_1 - \hat p_2$ | 0.266 − 0.155 = 0.111 | 463 / 537 |
| P4 | $\mu_1 - \mu_2$ | $\bar x_1 - \bar x_2$ | 46.50 − 37.10 = 9.41 años | 206 / 794 |

## 🎤 Qué decir en la exposición
> De los 1,000 clientes, 206 abandonaron: 0.206 es la estimación puntual de la proporción de abandono de los 10,000 clientes, unas 2,060 personas. Pero otra muestra daría otro número, y un solo número no dice qué tan cerca está del valor real. Por eso construimos intervalos de confianza.

## ❓ Preguntas del profesor
**¿Diferencia entre parámetro y estadístico?** El parámetro es de toda la población y no se conoce. El estadístico se calcula con la muestra y sirve para estimar el parámetro.

**¿Por qué no basta la estimación puntual?** Porque cambia de una muestra a otra y no dice cuánto se puede alejar del valor real.

---
# Sección 5 · Estimación por intervalo

## 📄 Qué dice el reporte: la receta común
Los 4 intervalos se arman igual:

$$\text{intervalo} = \text{estimación} \pm \underbrace{\text{valor crítico} \times \text{error estándar} \times \text{FCPF}}_{\text{margen de error}}$$

| Pieza | Qué es | Valor en el proyecto |
|---|---|---|
| **Error estándar** | Cuánto cambia la estimación de una muestra a otra | Cambia según la pregunta |
| **Valor crítico** | Cuántos errores estándar sumar y restar para acertar el 95 % de las veces | $z$ = 1.96 (proporciones); $t$ ≈ 1.97 (medias) |
| **FCPF** | Factor de corrección por población finita. Se usa cuando la muestra es más del 5 % de la población; la nuestra es el 10 % | $\sqrt{(N-n)/(N-1)}$ = 0.9487 |

**Desviación estándar contra error estándar:** la desviación estándar ($s$) mide cuánto varían **los clientes** uno por uno. El error estándar ($s/\sqrt n$) mide cuánto varía **el promedio** de una muestra a otra. Por eso es mucho más chico.

**¿Por qué t en P2 y P4?** No se conoce la desviación estándar de toda la población, sólo la de la muestra. La distribución t corrige esa duda extra. Con muestras grandes, t es casi igual a z (1.97 contra 1.96).

In [7]:
# QUÉ HACE ESTA CELDA: factor de corrección y valor z (comunes a los 4 intervalos)
fcpf = np.sqrt((N - n) / (N - 1))
z_95 = stats.norm.ppf(0.975)
print("FCPF:", round(fcpf, 4))
print("z para 95 %:", round(z_95, 4))

FCPF: 0.9487
z para 95 %: 1.96


## 5.1 Intervalo para una proporción (P1)

$$IC = \hat p \pm z \sqrt{\frac{\hat p(1-\hat p)}{n}} \cdot \text{FCPF}$$

**Supuestos:** muestra aleatoria (sí) y al menos 10 casos de cada resultado (206 abandonan y 794 permanecen: sí).

In [8]:
# QUÉ HACE ESTA CELDA: intervalo de P1 (reporte 5.1)
# error estándar de una proporción, sin corregir y corregido
error_estandar_p_sin_corregir = np.sqrt(p_hat * (1 - p_hat) / n)
error_estandar_p = error_estandar_p_sin_corregir * fcpf
# margen = valor crítico × error estándar
margen_p = z_95 * error_estandar_p
limite_inf_p = p_hat - margen_p
limite_sup_p = p_hat + margen_p
print("Error estándar sin corregir:", round(error_estandar_p_sin_corregir, 4))
print("Error estándar corregido:", round(error_estandar_p, 4))
print("Margen de error:", round(margen_p, 4))
print("IC 95 % para p:", round(limite_inf_p, 3), "a", round(limite_sup_p, 3))

Error estándar sin corregir: 0.0128
Error estándar corregido: 0.0121
Margen de error: 0.0238
IC 95 % para p: 0.182 a 0.23


**Número por número:** EE = √(0.206 × 0.794 / 1000) = 0.0128 → × 0.9487 = 0.0121 → margen = 1.96 × 0.0121 = 0.0238 → **(0.182, 0.230)**.

## 5.2 Intervalo para una media (P2)

$$IC = \bar x \pm t_{0.025,\,n-1} \frac{s}{\sqrt n} \cdot \text{FCPF}$$

**Supuesto de normalidad:** los saldos **no** tienen forma de campana (Gráfica 3: muchos en 0). El intervalo sigue siendo válido porque, con 206 clientes, **el promedio** sí tiene forma de campana (Teorema del Límite Central). El reporte lo comprueba con un bootstrap de 10,000 remuestras (Gráfica 6).

In [9]:
# QUÉ HACE ESTA CELDA: intervalo de P2 (reporte 5.2)
n_saldo = len(saldo_abandona)
# std() = desviación estándar: cuánto varían los saldos de cliente a cliente
desviacion_saldo = saldo_abandona.std()
# grados de libertad = n − 1; t.ppf da el valor crítico t
grados_libertad_saldo = n_saldo - 1
t_saldo = stats.t.ppf(0.975, grados_libertad_saldo)
# error estándar de un promedio: s / raíz de n, corregido
error_estandar_saldo = desviacion_saldo / np.sqrt(n_saldo) * fcpf
margen_saldo = t_saldo * error_estandar_saldo
limite_inf_saldo = media_saldo - margen_saldo
limite_sup_saldo = media_saldo + margen_saldo
print("n (clientes que abandonaron):", n_saldo)
print("Desviación estándar s (€):", round(desviacion_saldo, 2))
print("Grados de libertad:", grados_libertad_saldo)
print("Valor t:", round(t_saldo, 4))
print("Error estándar corregido (€):", round(error_estandar_saldo, 2))
print("Margen de error (€):", round(margen_saldo, 2))
print("IC 95 % para μ (€):", round(limite_inf_saldo, 2), "a", round(limite_sup_saldo, 2))

n (clientes que abandonaron): 206
Desviación estándar s (€): 60592.91
Grados de libertad: 205
Valor t: 1.9716
Error estándar corregido (€): 4005.26
Margen de error (€): 7896.79
IC 95 % para μ (€): 83664.74 a 99458.33


**Número por número:** EE = 60,592.91 / √206 = 4,221.71 → × 0.9487 = 4,005.26 → margen = 1.9716 × 4,005.26 = 7,896.79 → **(€83,665; €99,458)**.

**En Excel:** `=T.INV.2T(0.05, 205)` da 1.9716. Si se usa tabla impresa y no existe la fila 205, se usa la fila 120 (1.980) o ∞ (1.960); el margen cambia unos 80 €.

## 5.3 Intervalo para una diferencia de proporciones (P3)

$$IC = (\hat p_1 - \hat p_2) \pm z \sqrt{\frac{\hat p_1(1-\hat p_1)}{n_1} + \frac{\hat p_2(1-\hat p_2)}{n_2}} \cdot \text{FCPF}$$

**Supuestos:** grupos independientes (cada cliente es activo **o** inactivo) y al menos 10 casos de cada resultado en cada grupo (123/340 y 83/454: sí). Los errores de los dos grupos **se suman** dentro de la raíz.

In [10]:
# QUÉ HACE ESTA CELDA: intervalo de P3 (reporte 5.3)
n_inactivos = len(abandono_inactivos)
n_activos = len(abandono_activos)
# pieza de cada grupo: p(1 − p) / n
pieza_inactivos = p_inactivos * (1 - p_inactivos) / n_inactivos
pieza_activos = p_activos * (1 - p_activos) / n_activos
# se suman las piezas y se saca raíz
error_estandar_dif_p = np.sqrt(pieza_inactivos + pieza_activos) * fcpf
margen_dif_p = z_95 * error_estandar_dif_p
limite_inf_dif_p = dif_p - margen_dif_p
limite_sup_dif_p = dif_p + margen_dif_p
# tamaño del efecto: cuántas veces más abandonan los inactivos
riesgo_relativo = p_inactivos / p_activos
print("n inactivos:", n_inactivos)
print("n activos:", n_activos)
print("Error estándar corregido:", round(error_estandar_dif_p, 4))
print("Margen de error:", round(margen_dif_p, 4))
print("IC 95 % para p1 − p2:", round(limite_inf_dif_p, 3), "a", round(limite_sup_dif_p, 3))
print("Riesgo relativo:", round(riesgo_relativo, 2))

n inactivos: 463
n activos: 537
Error estándar corregido: 0.0245
Margen de error: 0.0479
IC 95 % para p1 − p2: 0.063 a 0.159
Riesgo relativo: 1.72


**Cómo se lee:** el intervalo **no contiene el 0** → la diferencia existe en la población. Si contuviera el 0, podría no haber diferencia.

## 5.4 Intervalo para una diferencia de medias (P4) · método de Welch

$$IC = (\bar x_1 - \bar x_2) \pm t_{0.025,\,gl} \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}} \cdot \text{FCPF}$$

**¿Por qué Welch?** No supone que los dos grupos tengan la misma dispersión. Si la tienen, da casi el mismo resultado que el método clásico; si no, sigue siendo correcto. Los grados de libertad ($gl$) los calcula la fórmula de Welch–Satterthwaite.

In [11]:
# QUÉ HACE ESTA CELDA: intervalo de P4 con Welch (reporte 5.4)
n_abandona = len(edad_abandona)
n_permanece = len(edad_permanece)
# var() = varianza = desviación estándar al cuadrado
varianza_abandona = edad_abandona.var()
varianza_permanece = edad_permanece.var()
# piezas de la fórmula: varianza / n de cada grupo
pieza_abandona = varianza_abandona / n_abandona
pieza_permanece = varianza_permanece / n_permanece
# grados de libertad de Welch–Satterthwaite
numerador_gl = (pieza_abandona + pieza_permanece) ** 2
denominador_gl = pieza_abandona ** 2 / (n_abandona - 1) + pieza_permanece ** 2 / (n_permanece - 1)
gl_welch = numerador_gl / denominador_gl
t_edad = stats.t.ppf(0.975, gl_welch)
error_estandar_edad = np.sqrt(pieza_abandona + pieza_permanece) * fcpf
margen_edad = t_edad * error_estandar_edad
limite_inf_edad = dif_edad - margen_edad
limite_sup_edad = dif_edad + margen_edad
print("Desviación estándar, abandona:", round(np.sqrt(varianza_abandona), 2))
print("Desviación estándar, permanece:", round(np.sqrt(varianza_permanece), 2))
print("Grados de libertad de Welch:", round(gl_welch, 1))
print("Valor t:", round(t_edad, 4))
print("Margen de error (años):", round(margen_edad, 2))
print("IC 95 % para μ1 − μ2 (años):", round(limite_inf_edad, 2), "a", round(limite_sup_edad, 2))

Desviación estándar, abandona: 9.24
Desviación estándar, permanece: 9.94
Grados de libertad de Welch: 338.6
Valor t: 1.967
Margen de error (años): 1.37
IC 95 % para μ1 − μ2 (años): 8.04 a 10.78


## Tabla 9 · Resumen de los intervalos

| Parámetro | Estimación | IC 95 % | Margen |
|---|---|---|---|
| $p$ | 0.206 | (0.182, 0.230) | ± 0.024 |
| $\mu$ saldo | €91,562 | (€83,665; €99,458) | ± €7,897 |
| $p_1 - p_2$ | 0.111 | (0.063, 0.159) | ± 0.048 |
| $\mu_1 - \mu_2$ edad | 9.41 años | (8.04, 10.78) | ± 1.37 |

## 🎤 Qué decir en la exposición
> Los cuatro intervalos se construyen igual: estimación más y menos el valor crítico por el error estándar. Como la muestra es el 10 % de la población, el error estándar se multiplica por 0.9487. Para las proporciones usamos z = 1.96; para los promedios, la distribución t, porque sólo conocemos la desviación estándar de la muestra. Para comparar edades usamos el método de Welch, que no supone varianzas iguales.

## ❓ Preguntas del profesor
**¿Qué es el error estándar?** Cuánto cambia la estimación de una muestra a otra. Para P1 vale 0.0128 (0.0121 corregido).

**¿De dónde sale 1.96?** En una distribución normal, el 95 % de los valores está a menos de 1.96 errores estándar del centro.

**¿Por qué el FCPF?** Porque la muestra es el 10 % de la población (más del 5 %). Si ya se revisó una parte grande del total, la estimación varía menos. Reduce el error estándar un 5 %.

**¿Por qué sirve la fórmula si los saldos no son normales?** Por el Teorema del Límite Central: con 206 clientes, el promedio tiene forma de campana aunque los datos no. Se comprobó con el bootstrap (Gráfica 6).

**¿Por qué el intervalo del saldo es tan ancho?** Porque sólo 206 clientes abandonaron y los saldos varían mucho ($s$ = €60,593).

---
# Sección 6 · Análisis e interpretación

## 6.1 ¿Qué significa cada intervalo?

| Pregunta | Conclusión literal |
|---|---|
| P1 | Con 95 % de confianza, entre el 18.2 % y el 23.0 % de los clientes abandona: entre unos 1,820 y 2,300 de los 10,000 |
| P2 | Con 95 % de confianza, el saldo promedio de quienes abandonan está entre €83,665 y €99,458 |
| P3 | Con 95 % de confianza, los inactivos abandonan entre 6.3 y 15.9 puntos porcentuales más. No contiene el 0. Es una **asociación, no una causa**: un cliente puede dejar de usar su cuenta porque ya planea irse |
| P4 | Con 95 % de confianza, quienes abandonan son entre 8.0 y 10.8 años mayores. No contiene el 0 |

**Qué significa el 95 %:** describe al **método**. Si se repitiera el muestreo muchas veces, 95 de cada 100 intervalos construidos así contendrían el valor real.

⚠️ **Error que no hay que cometer:** decir "hay 95 % de probabilidad de que $p$ esté en este intervalo". $p$ es un número fijo; lo que cambia de una muestra a otra es el intervalo.

### P4 también como prueba de hipótesis (por si lo pregunta el profesor)
- **H0:** en la población, la edad promedio de quienes abandonan es igual a la de quienes se quedan.
- **H1:** son distintas (dos colas, porque un intervalo al 95 % corresponde a una prueba de dos colas con α = 0.05).
- **α = 0.05.**

In [12]:
# QUÉ HACE ESTA CELDA: prueba t de Welch para P4 y tamaño del efecto
# ttest_ind compara dos promedios; equal_var=False es el método de Welch
prueba_edad = stats.ttest_ind(edad_abandona, edad_permanece, equal_var=False)
# d de Cohen: diferencia de medias ÷ desviación estándar combinada
suma_cuadrados = (n_abandona - 1) * varianza_abandona + (n_permanece - 1) * varianza_permanece
varianza_combinada = suma_cuadrados / (n_abandona + n_permanece - 2)
d_cohen = dif_edad / np.sqrt(varianza_combinada)
print("Estadístico t:", round(prueba_edad.statistic, 2))
print("p-valor:", prueba_edad.pvalue)
print("d de Cohen:", round(d_cohen, 2))

Estadístico t: 12.82
p-valor: 6.235668920121681e-31
d de Cohen: 0.96


**Conclusión:** Se rechaza H0 (p < 0.001 < 0.05): la edad promedio de los clientes que abandonan es mayor que la de los que se quedan. La diferencia está a casi 13 errores estándar del 0.

**Importancia práctica:** d = 0.96. A partir de 0.8 se considera un efecto grande. La diferencia no sólo es estadísticamente significativa: también es grande.

**Intervalo y prueba dicen lo mismo:** el intervalo (8.0, 10.8) no contiene el 0 ⇔ se rechaza H0 con α = 0.05.

## 6.2 Nivel de confianza: ¿por qué 95 % y qué pasaría con 99 %? (Gráfica 7)

In [13]:
# QUÉ HACE ESTA CELDA: intervalo de P1 con 90 %, 95 % y 99 %
z_90 = stats.norm.ppf(0.95)
z_99 = stats.norm.ppf(0.995)
margen_90 = z_90 * error_estandar_p
margen_99 = z_99 * error_estandar_p
print("z 90 %:", round(z_90, 3))
print("z 99 %:", round(z_99, 3))
print("IC 90 %:", round(p_hat - margen_90, 3), "a", round(p_hat + margen_90, 3))
print("IC 95 %:", round(limite_inf_p, 3), "a", round(limite_sup_p, 3))
print("IC 99 %:", round(p_hat - margen_99, 3), "a", round(p_hat + margen_99, 3))
print("Ancho 99 % ÷ ancho 95 %:", round(z_99 / z_95, 3))
print("Ancho 90 % ÷ ancho 95 %:", round(z_90 / z_95, 3))

z 90 %: 1.645
z 99 %: 2.576
IC 90 %: 0.186 a 0.226
IC 95 %: 0.182 a 0.23
IC 99 %: 0.175 a 0.237
Ancho 99 % ÷ ancho 95 %: 1.314
Ancho 90 % ÷ ancho 95 %: 0.839


- Más confianza → valor crítico más grande → intervalo **más ancho**: con 99 % es 31 % más ancho; con 90 %, 16 % más estrecho.
- Se eligió **95 %** porque equilibra precisión y confiabilidad. Los errores de esta decisión son económicos y corregibles: 99 % sacrificaría precisión sin necesidad y 90 % sería poco exigente con dinero de por medio.

## 6.3 ¿Qué sucede si aumenta el tamaño de muestra? (Gráfica 8)

In [14]:
# QUÉ HACE ESTA CELDA: margen de error de P1 con n = 1,000 y con n = 2,000
n_doble = 2000
fcpf_doble = np.sqrt((N - n_doble) / (N - 1))
margen_doble = z_95 * np.sqrt(p_hat * (1 - p_hat) / n_doble) * fcpf_doble
print("Margen con n = 1,000 (puntos %):", round(margen_p * 100, 1))
print("Margen con n = 2,000 (puntos %):", round(margen_doble * 100, 1))

Margen con n = 1,000 (puntos %): 2.4
Margen con n = 2,000 (puntos %): 1.6


- El margen tiene $\sqrt n$ en el denominador: **para reducirlo a la mitad hay que cuadruplicar la muestra**.
- Con n = 1,000 el margen fue ±2.4 puntos, mejor que el objetivo de ±3, porque la proporción observada (0.21) varía menos que el caso 0.5.
- Duplicar a 2,000 sólo lo baja a ±1.6 puntos, al doble de costo.

## 6.4 Comprobación con la población completa (Gráfica 9)

Como tenemos a los 10,000, se puede comprobar el método. Primero se simulan 1,000 muestras y se cuenta cuántos intervalos contienen la $p$ real:

In [15]:
# QUÉ HACE ESTA CELDA: simula 1,000 muestras de 1,000 clientes (igual que el cuaderno 03)
p_real = poblacion["abandono"].mean()
generador = np.random.default_rng(123)
# np.tile repite la columna de abandono 1,000 veces, una copia por fila
copias = np.tile(poblacion["abandono"].to_numpy(), (1000, 1))
# permuted revuelve el orden dentro de cada fila; las primeras 1,000 columnas son una muestra sin repetidos
barajadas = generador.permuted(copias, axis=1)
muestras_simuladas = barajadas[:, :n]
# proporción de abandono de cada muestra
p_simuladas = muestras_simuladas.mean(axis=1)
margenes_simulados = z_95 * np.sqrt(p_simuladas * (1 - p_simuladas) / n) * fcpf
inferiores = p_simuladas - margenes_simulados
superiores = p_simuladas + margenes_simulados
# True si el intervalo contiene la p real (& = "y")
contiene = (inferiores <= p_real) & (p_real <= superiores)
print("p real:", round(p_real, 4))
print("Intervalos que contienen la p real:", contiene.sum(), "de 1000")

p real: 0.2037
Intervalos que contienen la p real: 946 de 1000


In [16]:
# QUÉ HACE ESTA CELDA: compara los 4 intervalos del estudio con los valores reales
abandona_pob = poblacion["abandono"] == 1
permanece_pob = poblacion["abandono"] == 0
inactivo_pob = poblacion["miembro_activo"] == 0
activo_pob = poblacion["miembro_activo"] == 1
saldo_real = poblacion.loc[abandona_pob, "saldo"].mean()
p_inactivos_real = poblacion.loc[inactivo_pob, "abandono"].mean()
p_activos_real = poblacion.loc[activo_pob, "abandono"].mean()
dif_p_real = p_inactivos_real - p_activos_real
edad_abandona_real = poblacion.loc[abandona_pob, "edad"].mean()
edad_permanece_real = poblacion.loc[permanece_pob, "edad"].mean()
dif_edad_real = edad_abandona_real - edad_permanece_real
print("P1 real:", round(p_real, 3), "| ¿dentro?:", limite_inf_p <= p_real <= limite_sup_p)
print("P2 real (€):", round(saldo_real, 0), "| ¿dentro?:", limite_inf_saldo <= saldo_real <= limite_sup_saldo)
print("P3 real:", round(dif_p_real, 3), "| ¿dentro?:", limite_inf_dif_p <= dif_p_real <= limite_sup_dif_p)
print("P4 real (años):", round(dif_edad_real, 2), "| ¿dentro?:", limite_inf_edad <= dif_edad_real <= limite_sup_edad)

P1 real: 0.204 | ¿dentro?: True
P2 real (€): 91109.0 | ¿dentro?: True
P3 real: 0.126 | ¿dentro?: True
P4 real (años): 7.43 | ¿dentro?: False


| Comprobación | Resultado |
|---|---|
| Simulación | 946 de 1,000 intervalos (94.6 %) contienen la $p$ real. **El reporte dice 942 (94.2 %)**: la simulación se reprogramó con otro sorteo; los dos están cerca del 95 % |
| Intervalos del estudio | **3 de 4** contienen el valor real. P4 falló: real 7.4 años, intervalo (8.0, 10.8) |

**Por qué falló P4 (pregunta segura del profesor):** con 95 % de confianza se espera que cerca de 1 de cada 20 intervalos falle. Es el fallo que el método admite y muestra por qué se habla de confianza y no de certeza. La conclusión de fondo se mantiene: quienes abandonan son mayores (la diferencia real, 7.4 años, también es positiva).

## 6.5 Limitaciones

1. El banco no está identificado y no hay fechas: las conclusiones aplican a esta cartera.
2. El intervalo sólo cubre el error del muestreo, no errores de registro.
3. El estudio es observacional: P3 y P4 son asociaciones, no causas.
4. Las fórmulas usan aproximaciones normales; se comprobaron con bootstrap y simulación.
5. Cada intervalo tiene 95 % por separado; que los cuatro acierten a la vez tiene una confianza menor (al menos 80 %). En la validación, uno falló.

## 🎤 Qué decir en la exposición
> El 95 % es la tasa de acierto del método. Lo comprobamos: de 1,000 muestras simuladas, 946 intervalos contienen el valor real. De nuestros 4 intervalos, 3 acertaron; el de la edad falló, que es el tipo de fallo que admite el 95 %. Con 99 % los intervalos serían 31 % más anchos; para ganar precisión sin perder confianza hay que aumentar la muestra, y para reducir el margen a la mitad hay que cuadruplicarla.

---
# Sección 7 · Teoría de decisión

## 📄 Qué dice el reporte
**Decisión:** si fuéramos el área de retención, ¿a quién dirigiríamos una campaña (llamada de un asesor más una bonificación)?

| Opción | Descripción |
|---|---|
| A1 | No hacer campaña |
| A2 | Sólo a clientes inactivos |
| A3 | A todos los clientes |

**Supuestos (no vienen de los datos):** costo $c$ = €25 por cliente, efectividad $e$ = 25 %, valor de un cliente retenido $V$ = €500 al año.

**Beneficio por cliente contactado:** $B = p \cdot e \cdot V - c$. Conviene si $p$ supera el umbral $p^* = c / (e \cdot V)$.

**Ejemplo:** 100 clientes con $p$ = 0.20 → se iban 20 → se retiene el 25 % = 5 → 5 × €500 = €2,500 → contactar a 100 cuesta €2,500 → beneficio €0. Por eso 0.20 es el umbral.

In [17]:
# QUÉ HACE ESTA CELDA: umbral e intervalo de cada grupo (Tabla 10)
costo = 25
efectividad = 0.25
valor_cliente = 500
p_umbral = costo / (efectividad * valor_cliente)
# error estándar de la proporción de abandono de cada grupo
error_estandar_inactivos = np.sqrt(p_inactivos * (1 - p_inactivos) / n_inactivos) * fcpf
error_estandar_activos = np.sqrt(p_activos * (1 - p_activos) / n_activos) * fcpf
limite_inf_inactivos = p_inactivos - z_95 * error_estandar_inactivos
limite_sup_inactivos = p_inactivos + z_95 * error_estandar_inactivos
limite_inf_activos = p_activos - z_95 * error_estandar_activos
limite_sup_activos = p_activos + z_95 * error_estandar_activos
print("Umbral p*:", p_umbral)
print("Inactivos, IC 95 %:", round(limite_inf_inactivos, 3), "a", round(limite_sup_inactivos, 3))
print("Activos, IC 95 %:", round(limite_inf_activos, 3), "a", round(limite_sup_activos, 3))
print("¿Todo el IC de inactivos arriba de p*?:", limite_inf_inactivos > p_umbral)
print("¿Todo el IC de activos abajo de p*?:", limite_sup_activos < p_umbral)

Umbral p*: 0.2
Inactivos, IC 95 %: 0.227 a 0.304
Activos, IC 95 %: 0.126 a 0.184
¿Todo el IC de inactivos arriba de p*?: True
¿Todo el IC de activos abajo de p*?: True


| Grupo | IC 95 % | Clientes estimados | Comparación con 0.20 |
|---|---|---|---|
| Inactivos | (0.227, 0.304) | 4,630 | Todo el intervalo arriba: **conviene** |
| Activos | (0.126, 0.184) | 5,370 | Todo el intervalo abajo: **no conviene** |

El reporte escribe 0.228 como límite inferior de los inactivos; el valor exacto es 0.22749, que redondea a 0.227.

**Escenarios:** cada opción se evalúa con el límite inferior (pesimista), la estimación puntual y el límite superior (optimista). Así la incertidumbre del muestreo entra en la decisión.

In [18]:
# QUÉ HACE ESTA CELDA: matriz de decisión (Tabla 11) y los tres criterios
clientes_inactivos = N * n_inactivos / n
clientes_activos = N - clientes_inactivos
# beneficio total = (p · e · V − c) × clientes del grupo, en los tres escenarios
inactivos_pesimista = (limite_inf_inactivos * efectividad * valor_cliente - costo) * clientes_inactivos
inactivos_puntual = (p_inactivos * efectividad * valor_cliente - costo) * clientes_inactivos
inactivos_optimista = (limite_sup_inactivos * efectividad * valor_cliente - costo) * clientes_inactivos
activos_pesimista = (limite_inf_activos * efectividad * valor_cliente - costo) * clientes_activos
activos_puntual = (p_activos * efectividad * valor_cliente - costo) * clientes_activos
activos_optimista = (limite_sup_activos * efectividad * valor_cliente - costo) * clientes_activos
matriz = pd.DataFrame(
    {"Pesimista": [0, inactivos_pesimista, inactivos_pesimista + activos_pesimista],
     "Puntual": [0, inactivos_puntual, inactivos_puntual + activos_puntual],
     "Optimista": [0, inactivos_optimista, inactivos_optimista + activos_optimista]},
    index=["A1 · No hacer campaña", "A2 · Sólo inactivos", "A3 · Todos"])
# maximin: el peor resultado de cada opción
peor_caso = matriz.min(axis=1)
# arrepentimiento: lo mejor de cada escenario menos lo que da cada opción
arrepentimiento = matriz.max() - matriz
maximo_arrepentimiento = arrepentimiento.max(axis=1)
print("Peor caso de cada opción (€):")
print(peor_caso.round(0))
print("Máximo arrepentimiento de cada opción (€):")
print(maximo_arrepentimiento.round(0))
matriz.round(0)

Peor caso de cada opción (€):
A1 · No hacer campaña        0.0
A2 · Sólo inactivos      15910.0
A3 · Todos              -34061.0
dtype: float64
Máximo arrepentimiento de cada opción (€):
A1 · No hacer campaña    60090.0
A2 · Sólo inactivos          0.0
A3 · Todos               49971.0
dtype: float64


,Pesimista,Puntual,Optimista
A1 · No hacer campaña,0.0,0.0,0.0
A2 · Sólo inactivos,15910.0,38000.0,60090.0
A3 · Todos,-34061.0,7500.0,49061.0


In [19]:
# QUÉ HACE ESTA CELDA: sensibilidad y dinero en juego
# efectividad mínima: despeja e de p · e · V − c = 0 con el escenario pesimista
efectividad_minima = costo / (limite_inf_inactivos * valor_cliente)
print("Efectividad mínima (%):", round(efectividad_minima * 100, 1))
# depósitos que se llevan quienes abandonan = N × p × saldo promedio, en millones de €
dinero_inferior = N * limite_inf_p * limite_inf_saldo / 1000000
dinero_puntual = N * p_hat * media_saldo / 1000000
dinero_superior = N * limite_sup_p * limite_sup_saldo / 1000000
print("Dinero en juego, inferior (millones €):", round(dinero_inferior, 0))
print("Dinero en juego, puntual (millones €):", round(dinero_puntual, 0))
print("Dinero en juego, superior (millones €):", round(dinero_superior, 0))

Efectividad mínima (%): 22.0
Dinero en juego, inferior (millones €): 152.0
Dinero en juego, puntual (millones €): 189.0
Dinero en juego, superior (millones €): 229.0


| Criterio | Pregunta que responde | Resultado | Elige |
|---|---|---|---|
| Valor esperado | ¿Qué gana más si todo sale como se estimó? | A2: €38,000; A3: €7,500 | A2 |
| Mejor peor caso (maximin) | ¿Qué deja mejor resultado si todo sale mal? | A2: €15,910; A1: €0; A3: −€34,061 | A2 |
| Mínimo arrepentimiento | ¿Con qué me arrepentiría menos? | A2: €0 | A2 |

- **Sensibilidad (Gráfica 10):** A2 gana en todo el intervalo sólo si la efectividad es de al menos **22 %**. El supuesto (25 %) está por encima, pero cerca. La decisión resiste el azar del muestreo, pero depende de qué tan bien funcione la campaña.
- **Costo de equivocarse:** lanzar sin que convenga cuesta como máximo ≈ €116 mil (€25 × 4,630). No lanzar cuando convenía deja ir clientes que se llevan entre **€152 y €229 millones** en depósitos. No actuar es el error más caro.

**Recomendación:**
1. Campaña de retención **sólo a inactivos**.
2. No gastar en activos.
3. Empezar con un **piloto con grupo de control** (500 inactivos con campaña y 500 sin ella, sorteados) para medir la efectividad real.
4. Investigar el mayor abandono en Alemania.

## 🎤 Qué decir en la exposición
> Con un costo de €25, una efectividad de 25 % y un valor de €500 por cliente, la campaña gana dinero si el grupo abandona más del 20 %. Todo el intervalo de los inactivos, de 22.7 % a 30.4 %, está arriba de ese umbral, y todo el de los activos está abajo. Por eso los tres criterios eligen la campaña sólo a inactivos. El supuesto más débil es la efectividad: si baja de 22 %, la decisión podría cambiar. Por eso recomendamos empezar con un piloto.

## ❓ Preguntas del profesor
**¿Por qué usar el intervalo y no sólo la estimación?** Porque la estimación puntual ignora la incertidumbre. Si todo el intervalo está arriba del umbral, la campaña conviene incluso en el escenario pesimista.

**¿Los clientes se van porque son inactivos?** No se puede afirmar. El estudio es observacional. La inactividad identifica un grupo donde la campaña es rentable.

---
# Sección 8 · Decisiones metodológicas

⚠️ El documento pide responderlas **con sus propias palabras**. Abajo está la idea clave de cada una.

| # | Pregunta | Idea clave |
|---|---|---|
| 1 | ¿Por qué esa población? | La campaña aplica a toda la cartera: los 10,000 clientes de los tres países |
| 2 | ¿Por qué ese tamaño de muestra? | 95 %, ±3 puntos y p = 0.5 piden ≈ 965; se tomaron 1,000 por ser redondo y dar colchón a los subgrupos |
| 3 | ¿Cómo se seleccionaron los participantes? | Muestreo aleatorio simple sin reemplazo, semilla 2026; χ² confirmó que la muestra se parece a la población |
| 4 | ¿Por qué esas técnicas? | Cada una corresponde al tipo de variable: proporción (sí/no), media con t (saldo), diferencia de proporciones (dos grupos), Welch (edades sin suponer varianzas iguales) |
| 5 | ¿Por qué 95 %? | Equilibra precisión y confiabilidad; los errores son económicos y corregibles |
| 6 | ¿Qué supuestos? | Muestra aleatoria e independiente; muestras grandes (Teorema del Límite Central, comprobado con bootstrap y simulación); datos bien registrados; costo, efectividad y valor supuestos |
| 7 | ¿Qué limitaciones? | Banco no identificado y sin fechas; observacional (no prueba causas); no cubre errores de registro; confianza conjunta menor a 95 %; efectividad no medida |
| 8 | ¿Qué decisión puede tomar la organización? | Campaña sólo a inactivos, empezando con un piloto controlado |

---
# Conclusiones

> Con una muestra aleatoria de 1,000 clientes, el 10 % de la cartera, se estimaron cuatro valores de la población con su margen de error. Cerca de 1 de cada 5 clientes abandona el banco (entre 18.2 % y 23.0 %), y cada uno se lleva en promedio entre €83,665 y €99,458. El abandono es mayor entre los inactivos (entre 6.3 y 15.9 puntos más) y entre los clientes de mayor edad (entre 8.0 y 10.8 años más). Al comparar con los 10,000 clientes, 3 de los 4 intervalos acertaron, como se espera de un 95 % de confianza. Con estos intervalos y los costos supuestos, la decisión es lanzar una campaña de retención sólo para los inactivos, empezando con un piloto que mida su efectividad real.

# Guion de la exposición

| Orden | Sección | Qué decir | Números clave |
|---|---|---|---|
| 1 | 1 | Problema y 4 preguntas | 10,000 clientes; P1–P4 |
| 2 | 2 | Población, muestra y representatividad | N = 10,000; n = 1,000; semilla 2026; χ² p > 0.05 |
| 3 | 3 | Lo que se ve en la muestra | 20.6 % abandona; 26.6 % contra 15.5 %; Alemania ≈ 28 % |
| 4 | 4 | Estimaciones puntuales y por qué no bastan | 0.206; €91,562; 0.111; 9.4 años |
| 5 | 5 | La receta del intervalo y los 4 resultados | Tabla 9 |
| 6 | 6 | Qué significa el 95 % y la comprobación | 3 de 4; 946 de 1,000; 99 % → 31 % más ancho |
| 7 | 7 | La decisión | Umbral 0.20; A2; efectividad mínima 22 %; €152–229 millones |
| 8 | 8 y Conclusiones | Limitaciones y recomendación | Asociación no es causa; piloto con grupo de control |